# Calling Tool

## 1. Import necessary packages

In [1]:
from helpers import multiply, Tool, Agent

## 2. Create Tool abstraction

The way we've built by calling manually is prone to errors.

What we don't pass the correct type or miss one required field?

We would create an abstraction to make it easier to build a tool and call it.

The Tool class should have at least the following methods: 
- `__init__()` receiving the function and some logic to extract docs, arguments and their types
- `dict()` to return the json schema
- `__call__()` to enable the object instantiated to be callable. 

Example:
```python
class Tool:
    def __init__(self, func:Callable):
        self.func = func
    
    def dict(self):
        pass

    def __call__(self, *args, **kwargs):
        return self.func(*args, **kwargs)  

def my_func(arg1:int)->str:
    return "ok"

my_tool = Tool(my_func)
my_tool(arg1=1)
```



In [2]:
multiple_tool = Tool(multiply)

In [3]:
multiple_tool.dict()

{'type': 'function',
 'function': {'name': 'multiply',
  'description': 'Provides the result for multiplying two numbers',
  'parallel_tool_calls': False,
  'parameters': {'type': 'object',
   'properties': {'num1': {'type': 'number'}, 'num2': {'type': 'number'}},
   'required': ['num1', 'num2'],
   'additionalProperties': False},
  'strict': True}}

In [4]:
multiple_tool(25, 5)

125

## 3. Create Agent

In [11]:
agent = Agent(
    tools=[Tool(multiply)]
)

In [12]:
agent.invoke("what is 42 multiplied by 8?")

{'role': 'assistant',
 'content': '336\n\n42 multiplied by 8 equals 336.',
 'tool_calls': None}

In [13]:
agent.memory.get_messages()

[{'role': 'system',
  'content': "You're an helpful AI Agent, your role is Personal Assistant, and you need to Assist the user with their queries",
  'tool_calls': {}},
 {'role': 'user', 'content': 'what is 42 multiplied by 8?', 'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_2DJrJknkvmWZ1iDUfSOrztbu', function=Function(arguments='{"num1":42,"num2":8}', name='multiply'), type='function')]},
 {'role': 'tool',
  'content': '336',
  'tool_call_id': 'call_2DJrJknkvmWZ1iDUfSOrztbu'},
 {'role': 'assistant',
  'content': '336\n\n42 multiplied by 8 equals 336.',
  'tool_calls': None}]

In [21]:
agent.memory.reset()

In [22]:
agent.invoke("what is 34 multiplied by 18?")

{'role': 'assistant', 'content': '612', 'tool_calls': None}

In [23]:
agent.memory.get_messages()

[{'role': 'user', 'content': 'what is 34 multiplied by 18?', 'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_ciVc7VDgKnLwcE6vVpuWVkl1', function=Function(arguments='{"num1":34,"num2":18}', name='multiply'), type='function')]},
 {'role': 'tool',
  'content': '612',
  'tool_call_id': 'call_ciVc7VDgKnLwcE6vVpuWVkl1'},
 {'role': 'assistant', 'content': '612', 'tool_calls': None}]

In [39]:
agent.memory.reset()

In [40]:
agent.invoke("what is 34 multiply by ( 23 multiply by 12 )?")

{'role': 'assistant', 'content': 'The result is 9384.', 'tool_calls': None}

In [41]:
agent.memory.get_messages()

[{'role': 'user',
  'content': 'what is 34 multiply by ( 23 multiply by 12 )?',
  'tool_calls': {}},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_yYRnnN6czqsSjoqCAFKl94Tm', function=Function(arguments='{"num1":23,"num2":12}', name='multiply'), type='function')]},
 {'role': 'tool',
  'content': '276',
  'tool_call_id': 'call_yYRnnN6czqsSjoqCAFKl94Tm'},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_ckODAbp3vsTIMGkHrQGxzvi7', function=Function(arguments='{"num1":34,"num2":276}', name='multiply'), type='function')]},
 {'role': 'tool',
  'content': '9384',
  'tool_call_id': 'call_ckODAbp3vsTIMGkHrQGxzvi7'},
 {'role': 'assistant', 'content': 'The result is 9384.', 'tool_calls': None}]